# Auto-Detection of Distance Metric from CRS

This notebook demonstrates the `SpatialCoordinateValuesSelector` class introduced
in [issue #75](https://github.com/mllam/weather-model-graphs/issues/75).

The class wraps a ball-tree to provide efficient nearest-neighbour and radius
queries using the correct distance metric for the coordinate reference system (CRS):

| CRS type | Distance metric | When to use |
|---|---|---|
| Geographic (lat/lon) — e.g. `ccrs.PlateCarree()` | **Haversine** | Global / sparse grids |
| Projected — e.g. `ccrs.LambertConformal()` | **Euclidean** | Regional / LAM grids |

Distances returned by the haversine path are in **metres**; Euclidean distances are
in the same units as the input coordinates.

## 1. Setup and Imports

In [ ]:
import warnings

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np

import weather_model_graphs as wmg
from weather_model_graphs.spatial import SpatialCoordinateValuesSelector

print("weather_model_graphs imported OK")
print(f"SpatialCoordinateValuesSelector: {SpatialCoordinateValuesSelector}")

## 2. Euclidean Distance Queries (Projected CRS)

For a projected coordinate system the coordinates are Cartesian (e.g. metres).
We use the `"euclidean"` metric.

In [ ]:
rng = np.random.default_rng(42)

# 50 random Cartesian points in a 100 km × 100 km domain (units: metres)
n_pts = 50
projected_coords = rng.random((n_pts, 2)) * 1e5  # shape (50, 2)

sel_euc = SpatialCoordinateValuesSelector("euclidean", projected_coords)
print(f"metric: {sel_euc.distance_metric}")

# ---- k-nearest-to ----
query_pt = np.array([5e4, 5e4])  # centre of the domain
idxs, dists = sel_euc.k_nearest_to(query_pt, k=5)
print("\n5 nearest neighbours (Euclidean):")
for rank, (i, d) in enumerate(zip(idxs, dists), 1):
    print(f"  rank {rank}: index={i:3d}  dist={d/1e3:.2f} km  coord={projected_coords[i]}")

# ---- with_radius ----
radius_m = 2e4  # 20 km
idxs_r, dists_r = sel_euc.with_radius(query_pt, radius=radius_m)
print(f"\nPoints within {radius_m/1e3:.0f} km of centre: {len(idxs_r)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (highlight_idxs, title, radius) in zip(
    axes,
    [
        (idxs, "k=5 nearest neighbours", None),
        (idxs_r, f"within_radius = {radius_m/1e3:.0f} km", radius_m),
    ],
):
    ax.scatter(
        projected_coords[:, 0] / 1e3,
        projected_coords[:, 1] / 1e3,
        c="lightgrey", s=30, zorder=2, label="all points",
    )
    ax.scatter(
        projected_coords[highlight_idxs, 0] / 1e3,
        projected_coords[highlight_idxs, 1] / 1e3,
        c="steelblue", s=60, zorder=3, label="selected",
    )
    ax.scatter(*query_pt / 1e3, c="red", s=100, marker="*", zorder=4, label="query")
    if radius is not None:
        circle = plt.Circle(query_pt / 1e3, radius / 1e3, fill=False, color="red", lw=1.5, ls="--")
        ax.add_patch(circle)
    ax.set_aspect("equal")
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.set_title(title)
    ax.legend(loc="upper right")

fig.suptitle("Euclidean (projected CRS) distance queries", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Haversine Distance Queries (Geographic CRS)

For a geographic CRS the coordinates are longitude / latitude in degrees.
We use the `"haversine"` metric, and **all distances are returned in metres**.

In [ ]:
# 50 random lon/lat points over Europe (lon: -10 to 30, lat: 45 to 70)
geo_coords = np.column_stack([
    rng.uniform(-10, 30, n_pts),   # longitude
    rng.uniform(45, 70, n_pts),    # latitude
])

sel_hav = SpatialCoordinateValuesSelector("haversine", geo_coords)
print(f"metric: {sel_hav.distance_metric}")

# Known result: 1 degree of longitude at latitude 55° ≈ 63,800 m
# (cos(55°) * 111,195 m/deg ≈ 63,800 m)
query_geo = np.array([10.0, 55.0])  # lon=10°, lat=55°
idxs_h, dists_h = sel_hav.k_nearest_to(query_geo, k=5)
print("\n5 nearest neighbours (Haversine, distances in metres):")
for rank, (i, d) in enumerate(zip(idxs_h, dists_h), 1):
    print(f"  rank {rank}: index={i:3d}  dist={d/1e3:7.1f} km  lon={geo_coords[i,0]:.2f}° lat={geo_coords[i,1]:.2f}°")

# Radius query: 500 km around the query point
radius_500km = 500_000
idxs_hr, dists_hr = sel_hav.with_radius(query_geo, radius=radius_500km)
print(f"\nPoints within {radius_500km/1e3:.0f} km: {len(idxs_hr)}")

## 4. CRS-Based Automatic Metric Selection (`for_crs`)

The class-method `SpatialCoordinateValuesSelector.for_crs(crs, coords)` inspects
`crs.is_geographic` and picks the correct metric automatically — no manual choice needed.

In [ ]:
cases = {
    "PlateCarree (geographic)": ccrs.PlateCarree(),
    "Mercator (geographic)": ccrs.Mercator(),
    "LambertConformal (projected)": ccrs.LambertConformal(),
    "Stereographic (projected)": ccrs.Stereographic(),
    "Mollweide (projected)": ccrs.Mollweide(),
}

print(f"{'CRS':<40}  {'is_geographic':<15}  {'auto-selected metric'}")
print("-" * 75)
for name, crs in cases.items():
    sel = SpatialCoordinateValuesSelector.for_crs(crs, projected_coords)
    print(f"{name:<40}  {str(crs.is_geographic):<15}  {sel.distance_metric}")

## 5. Integration with Graph Connectivity

`SpatialCoordinateValuesSelector` is now used automatically inside
`wmg.create.create_all_graph_components()` for all nearest-neighbour and
radius queries.  When you supply `graph_crs`, the correct metric is chosen
for you — no other changes are required at the call site.

In [ ]:
# --- Projected CRS (euclidean) ---
lon = np.linspace(0.0, 9.0, 8)
lat = np.linspace(50.0, 58.0, 8)
lo, la = np.meshgrid(lon, lat)
lonlat_coords = np.column_stack([lo.ravel(), la.ravel()])

# Projected: convert to Oblique Mercator metres for demonstration
from pyproj import Transformer
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
x, y = transformer.transform(lonlat_coords[:, 0], lonlat_coords[:, 1])
proj_coords = np.column_stack([x, y])

# Build graph with projected CRS — euclidean metric is used automatically
import pyproj
web_mercator = pyproj.CRS("EPSG:3857")
G_proj = wmg.create.create_all_graph_components(
    coords=proj_coords,
    m2m_connectivity="flat",
    g2m_connectivity="nearest_neighbour",
    m2g_connectivity="nearest_neighbour",
    graph_crs=web_mercator,
)
lens_proj = [d["len"] for _, _, d in G_proj.edges(data=True) if "len" in d]
print(f"Projected graph: {G_proj.number_of_nodes()} nodes, {G_proj.number_of_edges()} edges")
print(f"Edge 'len' range: {min(lens_proj)/1e3:.1f} – {max(lens_proj)/1e3:.1f} km  (euclidean, metres)")

## 6. Rectilinear + Geographic CRS Warning

When a rectilinear mesh is overlaid on geographic (lat/lon) coordinates a
`UserWarning` is raised automatically, because equally-spaced lon/lat values are
**not** equally spaced on a sphere.

In [ ]:
# Trigger the warning intentionally by using a geographic CRS with a
# rectilinear (flat) mesh.  The haversine metric is still used for all
# neighbour queries; the warning just flags the mesh *layout*.

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    G_geo = wmg.create.create_all_graph_components(
        coords=lonlat_coords,
        m2m_connectivity="flat",
        g2m_connectivity="nearest_neighbour",
        m2g_connectivity="nearest_neighbour",
        graph_crs=ccrs.PlateCarree(),
    )

print(f"Warnings raised: {len(caught)}")
for w in caught:
    print(f"\n[{w.category.__name__}] {w.message}")

lens_geo = [d["len"] for _, _, d in G_geo.edges(data=True) if "len" in d]
print(f"\nGeographic graph: {G_geo.number_of_nodes()} nodes, {G_geo.number_of_edges()} edges")
print(f"Edge 'len' range: {min(lens_geo)/1e3:.1f} – {max(lens_geo)/1e3:.1f} km  (haversine, metres)")

## 7. Running the Test Suite

Run the new tests from the notebook.  All tests should pass.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_spatial_index.py", "-v", "--tb=short"],
    capture_output=True,
    text=True,
    cwd="..",  # run from repo root (parent of docs/)
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)